# channel-list-reverse-build — worked example 2: Build the generator Sequential by reversing the discriminator's channel list

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `channel-list-reverse-build`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Given the discriminator's `hidden_channels` (innermost to outermost), the generator is assembled by reversing that list and iterating consecutive `(in_c, out_c)` pairs. Each generator block upsamples with a `ConvTranspose2d` (stride 2 doubles spatial size), so reversing the list makes the generator the spatial inverse of the discriminator.

## Worked solution

We assemble an `nn.Sequential` generator from the discriminator's channel list.

1. **Reverse first.** `gen_channels = hidden_channels[::-1]`. The discriminator walked `[128, 256, 512]` from input toward the bottleneck; the generator must walk `[512, 256, 128]` from the bottleneck back out. Slice-reverse keeps the symmetry obvious in the code.
2. **Pair up.** `gen_pairs = list(zip(gen_channels[:-1], gen_channels[1:]))` gives `[(512,256),(256,128)]`.
3. **Per-pair block.** For each `(in_c, out_c)` we append a `ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False)`, then `BatchNorm2d(out_c)`, then `ReLU(inplace=True)`. `bias=False` because the following BatchNorm re-learns the bias as its beta; `stride=2` with `kernel=4, padding=1` exactly doubles H and W.
4. **Why reversing is essential.** If we iterated the un-reversed list, the generator's first conv would expect 128 input channels but the latent/bottleneck feeds 512 — a shape mismatch. Reversing aligns the generator's entry width with the discriminator's deepest width.
5. We verify by passing a 4-D tensor whose channel count equals the first reversed channel and confirming the spatial size doubled per block.

In [ ]:
import torch.nn as nn

def build_generator(hidden_channels):
    gen_channels = hidden_channels[::-1]
    gen_pairs = list(zip(gen_channels[:-1], gen_channels[1:]))
    blocks = []
    for in_c, out_c in gen_pairs:
        blocks.append(nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1, bias=False))
        blocks.append(nn.BatchNorm2d(out_c))
        blocks.append(nn.ReLU(inplace=True))
    return nn.Sequential(*blocks)

t.manual_seed(0)
hidden = [64, 128, 256, 512]
gen = build_generator(hidden)
x = t.randn(2, 512, 4, 4)  # enters at the reversed-first channel
out = gen(x)
print('num modules:', len(gen))
print('input  :', tuple(x.shape))
print('output :', tuple(out.shape))